# AMI Subset ASR-Diarization Alignment (Whisper + pyannote)

This notebook aligns Whisper ASR segments with pyannote diarization turns and computes per-segment features for candidate selection.

In [1]:
from __future__ import annotations

import json
import math
import random
import re
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ASR_DIR = PROJECT_ROOT / "data" / "asr_outputs" / "segments"
DIAR_DIR = PROJECT_ROOT / "data" / "diarization_outputs" / "json"
ALIGNED_DIR = PROJECT_ROOT / "data" / "aligned_chunks"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics"

ALIGNED_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[a-z0-9]+")

random.seed(0)

In [2]:
@dataclass
class AsrSegment:
    start: float
    end: float
    text: str
    avg_logprob: float
    no_speech_prob: float


@dataclass
class DiarSegment:
    start: float
    end: float
    speaker: str
    confidence: float
    seg_consistency: float
    overlap: float
    flip_rate: float


def load_asr_segments(path: Path) -> list[AsrSegment]:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    segments = []
    for row in payload.get("segments", []):
        segments.append(
            AsrSegment(
                start=float(row["start"]),
                end=float(row["end"]),
                text=str(row.get("text", "")),
                avg_logprob=float(row.get("avg_logprob", -10.0)),
                no_speech_prob=float(row.get("no_speech_prob", 0.0)),
            )
        )

    return segments


def load_diar_segments(path: Path) -> list[DiarSegment]:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    segments = []
    for row in payload.get("segments", []):
        segments.append(
            DiarSegment(
                start=float(row["start"]),
                end=float(row["end"]),
                speaker=str(row.get("speaker", "UNKNOWN")),
                confidence=float(row.get("confidence", 0.0)),
                seg_consistency=float(row.get("seg_consistency", 0.0)),
                overlap=float(row.get("overlap", 0.0)),
                flip_rate=float(row.get("flip_rate", 0.0)),
            )
        )

    return segments


def compute_overlap(start_a: float, end_a: float, start_b: float, end_b: float) -> float:
    return max(0.0, min(end_a, end_b) - max(start_a, start_b))


def normalize_logprobs(segments: list[AsrSegment]) -> dict[int, float]:
    if not segments:
        return {}

    values = [seg.avg_logprob for seg in segments]
    min_val = min(values)
    max_val = max(values)
    span = max_val - min_val

    normalized = {}
    for idx, seg in enumerate(segments):
        if span == 0:
            normalized[idx] = 1.0
        else:
            normalized[idx] = (seg.avg_logprob - min_val) / span

    return normalized


def tokenize(text: str) -> set[str]:
    return set(TOKEN_RE.findall(text.lower()))


def jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def compute_redundancy(asr_segments: list[AsrSegment], idx: int) -> float:
    current = tokenize(asr_segments[idx].text)
    neighbors = []
    if idx > 0:
        neighbors.append(tokenize(asr_segments[idx - 1].text))
    if idx + 1 < len(asr_segments):
        neighbors.append(tokenize(asr_segments[idx + 1].text))

    if not neighbors:
        return 0.0

    return max(jaccard(current, neighbor) for neighbor in neighbors)


def compute_turn_completeness(segment: AsrSegment) -> float:
    duration = max(0.0, segment.end - segment.start)
    duration_score = min(1.0, duration / 5.0)

    text = segment.text.strip()
    token_count = len(tokenize(text))
    if text.endswith((".", "?", "!")):
        punctuation_score = 1.0
    elif text.endswith(","):
        punctuation_score = 0.7
    else:
        punctuation_score = 0.85

    length_score = min(1.0, token_count / 6.0)
    speech_score = max(0.0, 1.0 - segment.no_speech_prob)

    return duration_score * punctuation_score * length_score * speech_score


def compute_diar_stability(overlaps: list[tuple[DiarSegment, float]]) -> float:
    if not overlaps:
        return 0.0

    total_overlap = sum(overlap for _, overlap in overlaps)
    if total_overlap <= 0.0:
        return 0.0

    speaker_overlap: dict[str, float] = {}
    for diar_seg, overlap in overlaps:
        speaker_overlap[diar_seg.speaker] = speaker_overlap.get(diar_seg.speaker, 0.0) + overlap

    dominant_overlap = max(speaker_overlap.values())
    speaker_consistency = dominant_overlap / total_overlap

    weighted_conf = 0.0
    weighted_consistency = 0.0
    weighted_non_overlap = 0.0
    weighted_non_flip = 0.0
    for diar_seg, overlap in overlaps:
        weight = overlap / total_overlap
        weighted_conf += diar_seg.confidence * weight
        weighted_consistency += diar_seg.seg_consistency * weight
        weighted_non_overlap += (1.0 - diar_seg.overlap) * weight
        weighted_non_flip += (1.0 - diar_seg.flip_rate) * weight

    stability = (
        speaker_consistency
        * weighted_conf
        * weighted_consistency
        * weighted_non_overlap
        * weighted_non_flip
    )
    return max(0.0, min(1.0, stability))

In [3]:
def align_asr_to_diar(asr_segments: list[AsrSegment], diar_segments: list[DiarSegment]) -> list[dict]:
    logprob_norm = normalize_logprobs(asr_segments)
    aligned_rows = []

    for idx, asr_seg in enumerate(asr_segments):
        overlaps = []
        for diar_seg in diar_segments:
            overlap = compute_overlap(
                asr_seg.start,
                asr_seg.end,
                diar_seg.start,
                diar_seg.end,
            )
            if overlap > 0.0:
                overlaps.append((diar_seg, overlap))

        if overlaps:
            overlaps.sort(
                key=lambda item: (
                    item[1],
                    item[0].confidence,
                    -item[0].start,
                ),
                reverse=True,
            )
            best_diar, best_overlap = overlaps[0]
            speaker = best_diar.speaker
            diar_turn_start = best_diar.start
            diar_turn_end = best_diar.end
        else:
            best_overlap = 0.0
            speaker = "UNKNOWN"
            diar_turn_start = math.nan
            diar_turn_end = math.nan

        segment_duration = max(0.0, asr_seg.end - asr_seg.start)
        if segment_duration > 0.0:
            purity = best_overlap / segment_duration
        else:
            purity = 0.0
        mix_penalty = 1.0 - purity

        asr_conf = logprob_norm.get(idx, 0.0)
        diar_stab = compute_diar_stability(overlaps)
        turn_comp = compute_turn_completeness(asr_seg)
        redund = compute_redundancy(asr_segments, idx)

        aligned_rows.append(
            {
                "start": asr_seg.start,
                "end": asr_seg.end,
                "text": asr_seg.text,
                "speaker": speaker,
                "ASRConf": asr_conf,
                "DiarStab": diar_stab,
                "TurnComp": turn_comp,
                "Redund": redund,
                "Purity": purity,
                "MixPenalty": mix_penalty,
                "diar_turn_start": diar_turn_start,
                "diar_turn_end": diar_turn_end,
                "diar_overlap_sec": best_overlap,
            }
        )

    return aligned_rows


def process_file(
    file_id: str,
    asr_dir: Path,
    diar_dir: Path,
    aligned_dir: Path,
    metrics_dir: Path,
) -> dict[str, float]:
    asr_path = asr_dir / f"{file_id}.json"
    diar_path = diar_dir / f"{file_id}.json"
    if not asr_path.exists() or not diar_path.exists():
        return {"processed": 0.0}

    asr_segments = load_asr_segments(asr_path)
    diar_segments = load_diar_segments(diar_path)
    aligned_rows = align_asr_to_diar(asr_segments, diar_segments)

    aligned_df = pd.DataFrame(aligned_rows)
    aligned_df.insert(0, "file_id", file_id)

    aligned_out = aligned_dir / f"{file_id}_aligned.csv"
    aligned_df.to_csv(aligned_out, index=False)

    metrics_df = aligned_df[
        [
            "file_id",
            "start",
            "end",
            "speaker",
            "ASRConf",
            "DiarStab",
            "TurnComp",
            "Redund",
            "Purity",
            "MixPenalty",
        ]
    ]
    metrics_out = metrics_dir / f"{file_id}_features.csv"
    metrics_df.to_csv(metrics_out, index=False)

    return {"processed": float(len(aligned_df))}


def collect_file_ids(asr_dir: Path, diar_dir: Path) -> list[str]:
    asr_ids = {path.stem for path in asr_dir.glob("*.json")}
    diar_ids = {path.stem for path in diar_dir.glob("*.json")}
    return sorted(asr_ids & diar_ids)

In [4]:
file_ids = collect_file_ids(ASR_DIR, DIAR_DIR)

processed = 0
for file_id in file_ids:
    result = process_file(file_id, ASR_DIR, DIAR_DIR, ALIGNED_DIR, METRICS_DIR)
    processed += int(result.get("processed", 0.0))

summary_df = pd.DataFrame({
    "metric": ["files", "segments"],
    "value": [len(file_ids), processed],
})
summary_df

,metric,value
0,files,12
1,segments,6405


## Retrieval + Reranking (Embeddings, Index, Evaluation)

In [5]:
import sys

!"{sys.executable}" -m ensurepip --upgrade
!"{sys.executable}" -m pip install --upgrade pip
!"{sys.executable}" -m pip install faiss-cpu

Looking in links: /var/folders/cg/1qj3kl_12w168jf9m109fgrr0000gn/T/tmpgcenzsiv


In [6]:
!"{sys.executable}" -m pip install sentence-transformers

In [7]:
import sys
import site

print("Python executable:", sys.executable)
print("User site:", site.getusersitepackages())

try:
    import faiss
    print("faiss imported:", faiss.__version__ if hasattr(faiss, "__version__") else "ok")
except Exception as exc:
    print("faiss import error:", repr(exc))

Python executable: /Users/adityagoyal/SRH/Thesis Research/thesis_code/.venv/bin/python
User site: /Users/adityagoyal/.local/lib/python3.12/site-packages
faiss imported: 1.14.2


In [8]:
import datetime
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd

try:
    import faiss
except ImportError as exc:
    raise ImportError("faiss not installed. Install faiss-cpu.") from exc

try:
    from sentence_transformers import SentenceTransformer
    import torch
except ImportError as exc:
    raise ImportError("sentence-transformers not installed. Install sentence-transformers.") from exc

RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval_results"
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_CACHE = METRICS_DIR / "chunk_embeddings_minilm.npz"
TOP_K = 10
BATCH_SIZE = 128

DEVICE = "mps" if "torch" in globals() and torch.backends.mps.is_available() else "cpu"

WEIGHTS = {
    "a": 1.0,
    "b": 0.3,
    "g": 0.3,
    "d": 0.2,
    "e": 0.1,
    "m": 0.4,
}

QUERIES_CSV = RETRIEVAL_DIR / "retrieval_queries.csv"
RUN_TAG = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

/Users/adityagoyal/SRH/Thesis Research/thesis_code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
@dataclass
class ChunkRecord:
    chunk_id: str
    audio_id: str
    start: float
    end: float
    text: str
    speaker_label: str
    ASRConf: float
    DiarStab: float
    TurnComp: float
    Redund: float
    MixPenalty: float
    Purity: float


def standardize_chunk_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {
        "speaker": "speaker_label",
        "audio_id": "audio_id",
        "file_id": "audio_id",
    }
    for src, dst in rename_map.items():
        if src in df.columns and dst not in df.columns:
            df = df.rename(columns={src: dst})
    if "speaker_label" not in df.columns:
        df["speaker_label"] = "UNKNOWN"
    if "audio_id" not in df.columns:
        df["audio_id"] = "UNKNOWN"
    if "chunk_id" not in df.columns:
        df["chunk_id"] = (
            df["audio_id"].astype(str)
            + "_"
            + df["start"].astype(float).round(3).astype(str)
            + "_"
            + df["end"].astype(float).round(3).astype(str)
        )

    numeric_defaults = {
        "ASRConf": 0.0,
        "DiarStab": 0.0,
        "TurnComp": 0.0,
        "Redund": 0.0,
        "MixPenalty": 0.0,
        "Purity": 0.0,
    }
    for col, default in numeric_defaults.items():
        if col not in df.columns:
            df[col] = default
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)

    if "text" not in df.columns:
        df["text"] = ""
    df["text"] = df["text"].astype(str)

    return df


def build_chunk_records(df: pd.DataFrame) -> list[ChunkRecord]:
    records = []
    for row in df.itertuples(index=False):
        records.append(
            ChunkRecord(
                chunk_id=str(row.chunk_id),
                audio_id=str(row.audio_id),
                start=float(row.start),
                end=float(row.end),
                text=str(row.text),
                speaker_label=str(row.speaker_label),
                ASRConf=float(row.ASRConf),
                DiarStab=float(row.DiarStab),
                TurnComp=float(row.TurnComp),
                Redund=float(row.Redund),
                MixPenalty=float(row.MixPenalty),
                Purity=float(row.Purity),
            )
        )
    return records


def load_aligned_chunks(aligned_dir: Path) -> tuple[pd.DataFrame, list[ChunkRecord]]:
    paths = sorted(aligned_dir.glob("*_aligned.csv"))
    if not paths:
        raise FileNotFoundError(f"No aligned chunks found in {aligned_dir}")

    frames = [pd.read_csv(path) for path in paths]
    df = pd.concat(frames, ignore_index=True)
    df = standardize_chunk_columns(df)

    required_cols = ["chunk_id", "audio_id", "start", "end", "text", "speaker_label"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    records = build_chunk_records(df)
    return df, records

In [10]:
def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return matrix / norms


def build_embeddings(texts: Iterable[str], model: SentenceTransformer, batch_size: int = 64) -> np.ndarray:
    vectors = model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    return vectors


def load_embedding_cache(path: Path) -> tuple[list[str], np.ndarray] | None:
    if not path.exists():
        return None
    payload = np.load(path, allow_pickle=True)
    ids = payload.get("ids")
    vectors = payload.get("vectors")
    if ids is None or vectors is None:
        return None
    return list(ids.tolist()), vectors


def save_embedding_cache(path: Path, ids: list[str], vectors: np.ndarray) -> None:
    np.savez(path, ids=np.array(ids), vectors=vectors)


def build_embeddings_cached(
    df: pd.DataFrame,
    model: SentenceTransformer,
    cache_path: Path,
    batch_size: int = 64,
) -> tuple[np.ndarray, list[str]]:
    ids = df["chunk_id"].astype(str).tolist()
    cached = load_embedding_cache(cache_path)
    if cached is not None:
        cached_ids, cached_vectors = cached
        if cached_ids == ids and cached_vectors.shape[0] == len(ids):
            return cached_vectors, ids

    vectors = build_embeddings(df["text"].tolist(), model, batch_size=batch_size)
    vectors = normalize_rows(vectors)
    save_embedding_cache(cache_path, ids, vectors)
    return vectors, ids


def build_vector_index(vectors: np.ndarray) -> faiss.IndexFlatIP:
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors.astype(np.float32))
    return index

In [11]:
def retrieve_candidates(
    query: str,
    top_k: int,
    model: SentenceTransformer,
    index: faiss.IndexFlatIP,
    df: pd.DataFrame,
    id_map: list[str],
) -> pd.DataFrame:
    query_vec = model.encode([query], convert_to_numpy=True, normalize_embeddings=False)
    query_vec = normalize_rows(query_vec)
    scores, indices = index.search(query_vec.astype(np.float32), top_k)
    scores = scores.flatten()
    indices = indices.flatten()

    rows = []
    for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
        if idx < 0 or idx >= len(id_map):
            continue
        chunk_id = id_map[idx]
        row = df.loc[df["chunk_id"] == chunk_id].iloc[0].to_dict()
        row["semantic_score"] = float(score)
        row["rank"] = rank
        rows.append(row)

    return pd.DataFrame(rows)


def rerank_candidates(candidates: pd.DataFrame, weights: dict[str, float]) -> pd.DataFrame:
    if candidates.empty:
        return candidates.copy()

    df = candidates.copy()
    for col in ["ASRConf", "DiarStab", "TurnComp", "Redund", "MixPenalty", "semantic_score"]:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    df["final_score"] = (
        weights.get("a", 1.0) * df["semantic_score"]
        + weights.get("b", 0.0) * df["ASRConf"]
        + weights.get("g", 0.0) * df["DiarStab"]
        + weights.get("d", 0.0) * df["TurnComp"]
        + weights.get("e", 0.0) * df["Redund"]
        - weights.get("m", 0.0) * df["MixPenalty"]
    )
    df = df.sort_values("final_score", ascending=False).reset_index(drop=True)
    df["rerank"] = df.index + 1
    return df

In [12]:
def _compute_metrics_for_query(relevance: list[int], total_relevant: int) -> dict[str, float]:
    if not relevance:
        return {"mrr": 0.0, "recall": 0.0, "precision": 0.0, "ndcg": 0.0}

    first_hit = next((idx + 1 for idx, rel in enumerate(relevance) if rel > 0), None)
    mrr = 1.0 / first_hit if first_hit is not None else 0.0

    hits = sum(1 for rel in relevance if rel > 0)
    precision = hits / len(relevance)
    recall = hits / total_relevant if total_relevant > 0 else 0.0

    dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
    ideal_hits = min(total_relevant, len(relevance))
    idcg = sum(1.0 / np.log2(idx + 2) for idx in range(ideal_hits))
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {"mrr": mrr, "recall": recall, "precision": precision, "ndcg": ndcg}


def evaluate_retrieval(
    queries_df: pd.DataFrame,
    chunks_df: pd.DataFrame,
    model: SentenceTransformer,
    index: faiss.IndexFlatIP,
    id_map: list[str],
    weights: dict[str, float],
    top_k: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required = {"query_id", "query_text", "chunk_id", "relevance"}
    missing = required - set(queries_df.columns)
    if missing:
        raise ValueError(f"Query CSV missing columns: {sorted(missing)}")

    queries_df = queries_df.copy()
    queries_df["relevance"] = pd.to_numeric(queries_df["relevance"], errors="coerce").fillna(0).astype(int)
    query_texts = queries_df.drop_duplicates("query_id")[["query_id", "query_text"]]

    results_rows = []
    metrics_rows = []
    for row in query_texts.itertuples(index=False):
        query_id = row.query_id
        query_text = row.query_text
        relevant = queries_df[(queries_df["query_id"] == query_id) & (queries_df["relevance"] > 0)][
            "chunk_id"
        ].astype(str).tolist()
        relevant_set = set(relevant)

        candidates = retrieve_candidates(query_text, top_k, model, index, chunks_df, id_map)
        reranked = rerank_candidates(candidates, weights)
        reranked["query_id"] = query_id
        reranked["query_text"] = query_text
        reranked["relevance"] = reranked["chunk_id"].astype(str).isin(relevant_set).astype(int)
        results_rows.append(reranked)

        relevance_list = reranked["relevance"].tolist()
        metrics = _compute_metrics_for_query(relevance_list, len(relevant_set))
        metrics_rows.append({"query_id": query_id, **metrics})

    results_df = pd.concat(results_rows, ignore_index=True) if results_rows else pd.DataFrame()
    metrics_df = pd.DataFrame(metrics_rows)
    summary_df = metrics_df.mean(numeric_only=True).to_frame().T
    summary_df.insert(0, "metric", "mean")
    return results_df, metrics_df, summary_df


def save_results(
    results_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    retrieval_dir: Path,
    metrics_dir: Path,
    run_tag: str,
) -> None:
    retrieval_path = retrieval_dir / f"retrieval_results_{run_tag}.csv"
    metrics_path = metrics_dir / f"retrieval_metrics_{run_tag}.csv"
    summary_path = metrics_dir / f"retrieval_summary_{run_tag}.csv"
    results_df.to_csv(retrieval_path, index=False)
    metrics_df.to_csv(metrics_path, index=False)
    summary_df.to_csv(summary_path, index=False)

In [13]:
aligned_df, chunk_records = load_aligned_chunks(ALIGNED_DIR)
model = SentenceTransformer(EMBED_MODEL_NAME, device='mps')
embeddings, id_map = build_embeddings_cached(
    aligned_df,
    model,
    EMBED_CACHE,
    batch_size=BATCH_SIZE,
 )
index = build_vector_index(embeddings)

aligned_df.head()

Batches: 100%|██████████| 51/51 [00:11<00:00,  4.38it/s]


,audio_id,start,end,text,speaker_label,ASRConf,DiarStab,TurnComp,Redund,Purity,MixPenalty,diar_turn_start,diar_turn_end,diar_overlap_sec,chunk_id
0,095dc90b898b,0.0,28.0,"Yep, as soon as I get this.",SPEAKER_02,0.000000,1.0,0.282971,0.000000,0.151875,0.848125,14.138469,18.390969,4.252500,095dc90b898b_0.0_28.0
1,095dc90b898b,28.0,28.5,OK.,SPEAKER_02,0.000000,1.0,0.004716,0.000000,1.000000,0.000000,27.942219,28.819719,0.500000,095dc90b898b_28.0_28.5
2,095dc90b898b,30.0,36.0,This is our last meeting.,SPEAKER_02,0.938914,1.0,0.798318,0.066667,0.279839,0.720161,34.320969,41.222844,1.679031,095dc90b898b_30.0_36.0
3,095dc90b898b,36.0,39.0,I'll go ahead and go through the minutes from ...,SPEAKER_02,0.938914,1.0,0.574789,0.187500,1.000000,0.000000,34.320969,41.222844,3.000000,095dc90b898b_36.0_39.0
4,095dc90b898b,39.0,47.0,And then we'll have the prototype presentation.,SPEAKER_02,0.938914,1.0,0.957981,0.187500,0.436641,0.563359,41.239719,44.732844,3.493125,095dc90b898b_39.0_47.0


In [14]:
if not QUERIES_CSV.exists():
    print(
        f"Query CSV not found at {QUERIES_CSV}. "
        "Create it with columns: query_id, query_text, chunk_id, relevance."
    )
else:
    queries_df = pd.read_csv(QUERIES_CSV)
    results_df, metrics_df, summary_df = evaluate_retrieval(
        queries_df,
        aligned_df,
        model,
        index,
        id_map,
        WEIGHTS,
        TOP_K,
    )
    save_results(results_df, metrics_df, summary_df, RETRIEVAL_DIR, METRICS_DIR, RUN_TAG)
    summary_df

Query CSV not found at /Users/adityagoyal/SRH/Thesis Research/thesis_code/data/retrieval_results/retrieval_queries.csv. Create it with columns: query_id, query_text, chunk_id, relevance.


In [15]:
example_query = "What are the key design features discussed for the remote control?"
candidates = retrieve_candidates(example_query, TOP_K, model, index, aligned_df, id_map)
reranked = rerank_candidates(candidates, WEIGHTS)
reranked.head(10)

,audio_id,start,end,text,speaker_label,ASRConf,DiarStab,TurnComp,Redund,Purity,MixPenalty,diar_turn_start,diar_turn_end,diar_overlap_sec,chunk_id,semantic_score,rank,final_score,rerank
0,8dcfe0e2e1ed,1214.00,1223.00,"As a user interface designer, I did a little r...",SPEAKER_03,0.867020,1.000000,0.999359,0.103448,1.000000,0.000000,1213.680969,1223.114094,9.000000,8dcfe0e2e1ed_1214.0_1223.0,0.804872,2,1.575195,1
1,97d6c569ebcf,312.52,317.52,Let's just throw out some ideas of what kind o...,SPEAKER_03,0.931576,1.000000,0.810537,0.260870,1.000000,0.000000,309.636594,326.494719,5.000000,97d6c569ebcf_312.52_317.52,0.710391,8,1.478059,2
2,ffdef30c3b55,405.36,407.36,What does the remote control do?,SPEAKER_01,1.000000,0.978645,0.381407,0.571429,1.000000,0.000000,392.324094,420.657219,2.000000,ffdef30c3b55_405.36_407.36,0.728668,6,1.455686,3
3,97d6c569ebcf,149.08,153.08,The functional design is looking at the techni...,SPEAKER_02,0.868398,1.000000,0.764022,0.150000,1.000000,0.000000,139.587219,169.995969,4.000000,97d6c569ebcf_149.08_153.08,0.719640,7,1.447964,4
4,3f83a5aca14d,741.00,743.00,of remote controls. It's just simple,SPEAKER_02,0.875907,1.000000,0.339263,0.250000,1.000000,0.000000,739.324719,756.537219,2.000000,3f83a5aca14d_741.0_743.0,0.753810,3,1.409435,5
5,3f83a5aca14d,393.00,395.00,remote controls.,SPEAKER_01,0.997496,1.000000,0.131308,0.000000,1.000000,0.000000,359.586594,431.406594,2.000000,3f83a5aca14d_393.0_395.0,0.702264,9,1.327775,6
6,3f83a5aca14d,823.00,825.00,of remote controls.,SPEAKER_02,0.774555,1.000000,0.199546,0.000000,0.898297,0.101703,812.241594,824.796594,1.796594,3f83a5aca14d_823.0_825.0,0.737338,5,1.268933,7
7,418beed50cb5,745.58,747.30,that you can customize your remote control.,SPEAKER_00,0.913467,0.000000,0.343842,0.058824,1.000000,0.000000,739.155969,755.339094,1.720000,418beed50cb5_745.58_747.3,0.752421,4,1.101112,8
8,1ad6824062ea,534.00,538.00,functionalities. So this is one of the very im...,SPEAKER_03,0.982751,0.043781,0.679270,0.333333,0.572258,0.427742,535.710969,539.372844,2.289031,1ad6824062ea_534.0_538.0,0.701879,10,1.007929,9
9,1ad6824062ea,804.00,812.00,of the remote controller design. I have just b...,SPEAKER_02,0.926620,0.041148,0.739495,0.105263,0.324844,0.675156,806.976594,809.575344,2.598750,1ad6824062ea_804.0_812.0,0.819589,1,0.998282,10
